# 2.4 — MyoReflex: spinal-feedback walking

This notebook runs the **Song-Geyer** reflex controller on `myoLegWalk-v0`. Muscle excitations come from spinal feedback (length, force, and ground load), not from a learned policy.

**What you'll learn:**
- How to load a 46-parameter gain vector
- How `MyoLegReflex` wraps a CPU Gymnasium env and steps the reflex
- How to render an offscreen rollout

**Prerequisites:** Completed notebook 1.1 · run this notebook from `tutorials/` or `tutorials/files/2.4/`

> Citation: Song, S., Geyer, H. (2015). A neural circuitry that emphasizes spinal feedback generates diverse behaviours of human locomotion. *The Journal of Physiology* 593(16), 3493-3511. [doi:10.1113/JP270228](https://doi.org/10.1113/JP270228). The wrapper maps 70 of the 80 MyoLeg muscles onto the original 11-per-leg groups (edl, ehl,
  fdl, fhl and tfl are left unactuated). The model will fall down; it is not a fully stabilized 3D gait and the shipped gains require reoptimization.


In [ ]:
import os; _nb = globals().get("__vsc_ipynb_file__"); _nb and os.chdir(os.path.dirname(_nb))  # VS Code: work relative to this notebook

import sys
from pathlib import Path

import numpy as np

from myosuite.utils.video_io import show_video, write_video

_here = Path.cwd()
if (_here / "reflex_ctr_interface.py").exists():
    sys.path.insert(0, str(_here))
elif (_here / "files/2.4" / "reflex_ctr_interface.py").exists():
    sys.path.insert(0, str(_here / "files/2.4"))
    _here = _here / "files/2.4"
else:
    raise FileNotFoundError(
        "Run from tutorials/ or tutorials/files/2.4/ so reflex_ctr_interface.py is importable."
    )

import reflex_ctr_interface


Load `baseline_params.txt`, reset the walker, and roll out. `normalize_act=False` so the reflex can write raw `[0, 1]` excitations. Rendering uses Gymnasium `rgb_array` (free camera).

In [ ]:
sim_time = 2  # seconds; the rollout ends early when the model falls
dt = 0.01
steps = int(sim_time / dt)
frames = []

params = np.loadtxt(_here / "baseline_params.txt")

Myo_env = reflex_ctr_interface.MyoLegReflex(sim_time=sim_time, seed=0)
Myo_env.reset()
Myo_env.set_control_params(params)

for timestep in range(steps):
    frame = Myo_env.env.render()
    if frame is not None:
        frames.append(frame)
    _, is_done, _, _ = Myo_env.run_reflex_step()
    if is_done:
        print(f"Stopped at step {timestep} ({timestep * dt:.2f} s)")
        break

Myo_env.env.close()

out_mp4 = _here / "MyoReflex_output.mp4"
if frames:
    write_video(out_mp4, np.asarray(frames), fps=round(1 / dt), outputdict={"-pix_fmt": "yuv420p"})
    print(f"Wrote {out_mp4} ({len(frames)} frames)")


In [ ]:
show_video(str(out_mp4))
